In [57]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf

In [58]:
import pandas as pd

# --------------------------------------------------
# File paths
# --------------------------------------------------

main_file = "selections.csv"
short_file = "selec_long_neg_precise.csv"
output_file = "selecs_positive_test_samedate.csv"


# --------------------------------------------------
# Load files
# --------------------------------------------------

df = pd.read_csv(main_file)
df["site.id"] = df["deploy.id"].astype(str).str.split("_").str[0]
df["duration"] = df["end"] - df["start"]
        

In [59]:
df["date"] = pd.to_datetime(
    df["sound.files"].str.extract(r"^(\d{8})")[0],
    format="%Y%m%d"
).dt.date

df["start.time"] = pd.to_datetime(
    df["sound.files"].str.extract(r"_(\d{6})\.")[0],
    format="%H%M%S"
).dt.time

df_neg = pd.read_csv(short_file)

In [60]:
df.head(3)

,selec,deploy.id,sound.files,view,channel,start,end,bottom.freq,top.freq,species.code,common.name,annotation,recording.id,site.id,duration,date,start.time
0,3,14569_1_2020_06_11_FVA133,20200611_191500.WAV,Spectrogram 1,1,46.570455,49.570455,5.3333,8.6974,eurwoo,Eurasian Woodcock,whistle,11599,14569,3.0,2020-06-11,19:15:00
1,2,14569_1_2020_06_11_FVA133,20200611_191500.WAV,Spectrogram 1,1,58.520463,61.520463,4.5538,11.9795,eurwoo,Eurasian Woodcock,whistle,11599,14569,3.0,2020-06-11,19:15:00
2,7,14569_1_2020_06_11_FVA133,20200611_191500.WAV,Spectrogram 1,1,64.378955,67.378955,4.6769,8.2462,eurwoo,Eurasian Woodcock,whistle,11599,14569,3.0,2020-06-11,19:15:00


In [61]:
df_neg = df_neg.merge(
    df,
    on="selec",
    how="left"
)

In [62]:
df_neg.head()

,selec,start_x,start.1,start.2,start.3,start.4,start.5,start.6,start.7,start.8,...,bottom.freq,top.freq,species.code,common.name,annotation,recording.id,site.id,duration,date,start.time
0,5058,0.0,3.50,7.50,11.2,14.7,18.2,22.15,30.20,35.0,...,2.4207,18.5720,eurwoo,Eurasian Woodcock,woodcock,15606,43791,38.62755,2020-06-27,20:25:14
1,5082,5.4,20.36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.6091,18.4971,eurwoo,Eurasian Woodcock,woodcock,15620,43791,25.48504,2020-06-28,19:25:02
2,5086,14.9,18.11,21.30,22.2,NaN,NaN,NaN,NaN,NaN,...,1.6971,15.9172,eurwoo,Eurasian Woodcock,woodcock,15621,43791,25.33770,2020-06-28,19:35:04
3,5095,10.0,18.20,21.70,25.1,NaN,NaN,NaN,NaN,NaN,...,1.1976,15.4517,eurwoo,Eurasian Woodcock,woodcock,15645,43791,32.72442,2020-06-29,20:15:12
4,5239,0.0,3.60,7.15,10.6,14.1,17.6,21.00,24.55,25.3,...,3.2112,23.3215,eurwoo,Eurasian Woodcock,woodcock,15666,43791,28.39083,2020-06-30,20:25:14


In [63]:
df_neg.columns

Index(['selec', 'start_x', 'start.1', 'start.2', 'start.3', 'start.4',
       'start.5', 'start.6', 'start.7', 'start.8', 'deploy.id', 'sound.files',
       'view', 'channel', 'start_y', 'end', 'bottom.freq', 'top.freq',
       'species.code', 'common.name', 'annotation', 'recording.id', 'site.id',
       'duration', 'date', 'start.time'],
      dtype='object')

In [64]:
df_neg = df_neg.drop(columns=[
    'start_y',
    'end',
    'species.code',
    'common.name',
    'bottom.freq',
    'top.freq'
])

df_neg = df_neg.rename(columns={'start_x': 'start.0'})

In [65]:
df.columns

Index(['selec', 'deploy.id', 'sound.files', 'view', 'channel', 'start', 'end',
       'bottom.freq', 'top.freq', 'species.code', 'common.name', 'annotation',
       'recording.id', 'site.id', 'duration', 'date', 'start.time'],
      dtype='object')

In [66]:
df_neg.loc[:, 'start.0':'start.8'].notna().sum()

start.0    47
start.1    44
start.2    39
start.3    37
start.4    19
start.5     9
start.6     4
start.7     2
start.8     2
dtype: int64

In [67]:
df_neg["site.id"].value_counts()

43791    37
45967     8
14569     1
49006     1
Name: site.id, dtype: int64

In [68]:
df_neg["recording.id"].value_counts()

15645    4
15620    4
15646    3
15666    3
15623    2
15621    2
15664    2
19668    2
15665    2
15663    2
15648    2
15649    2
15605    2
19670    2
15624    1
11602    1
15622    1
15588    1
15606    1
15628    1
19671    1
15607    1
15604    1
19666    1
19669    1
20631    1
19145    1
Name: recording.id, dtype: int64

In [69]:
selec_ids = df_neg["selec"].unique()
recording_ids = df_neg["recording.id"].unique()
deploy_ids = df_neg["deploy.id"].unique()
dates = df_neg["date"].unique()
site_ids = df_neg["site.id"].unique()

In [70]:
df.date.nunique()

50

In [71]:
dates

array([datetime.date(2020, 6, 27), datetime.date(2020, 6, 28),
       datetime.date(2020, 6, 29), datetime.date(2020, 6, 30),
       datetime.date(2021, 5, 28), datetime.date(2021, 5, 19),
       datetime.date(2020, 6, 11), datetime.date(2020, 6, 19)],
      dtype=object)

In [72]:
df_pos = df[
    ~df["selec"].isin(selec_ids)
].copy()

In [73]:
df_pos_dates = df_pos[df_pos["date"].isin(dates)].copy()
df_pos_recording = df_pos[df_pos["recording.id"].isin(recording_ids)].copy()
df_pos_site = df_pos[df_pos["site.id"].isin(site_ids)].copy()

In [74]:
df_pos_recording["recording.id"].nunique()

6

In [75]:
len(df_pos_recording), len(df_pos_dates), len(df_pos_site)

(20, 401, 1165)

In [76]:
len(site_ids)

4

In [77]:
df_pos_site_date = df_pos_dates[df_pos_dates["site.id"].isin(site_ids)]

In [78]:
len(df_pos_site_date)

247

In [79]:
df_pos.

SyntaxError: invalid syntax (3850753541.py, line 1)

In [ ]:
256

256

In [ ]:
#df_pos_site_date.to_csv("selecs_samedate_samesite_precise.csv", index=False)